# Apache Flink

Apache Flink is a distributed stream processing framework for stateful computations over unbounded and bounded data streams, providing high-throughput, low-latency processing with exactly-once semantics.

## Core Concepts

### DataStream API

DataStream API is the core abstraction for processing unbounded streams of data.

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.common.typeinfo import Types

env = StreamExecutionEnvironment.get_execution_environment()

# Create DataStream from collection
data_stream = env.from_collection(
    collection=[(1, 'Alice'), (2, 'Bob'), (3, 'Charlie')],
    type_info=Types.ROW([Types.INT(), Types.STRING()])
)

# Transform stream
result = data_stream.map(lambda x: (x[0], x[1].upper()))

# Print results
result.print()

# Execute
env.execute("DataStream Example")
```

### Operators

Operators are transformations applied to DataStreams.

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.common.typeinfo import Types

env = StreamExecutionEnvironment.get_execution_environment()

# Source
data_stream = env.from_collection([1, 2, 3, 4, 5])

# Map: transform each element
mapped = data_stream.map(lambda x: x * 2)

# Filter: select elements
filtered = data_stream.filter(lambda x: x % 2 == 0)

# FlatMap: produce zero or more outputs per input
flat_mapped = data_stream.flat_map(
    lambda x: [(x, x * 2), (x, x * 3)],
    result_type=Types.TUPLE([Types.INT(), Types.INT()])
)

# KeyBy: partition by key
keyed = data_stream.map(
    lambda x: (x % 2, x),
    output_type=Types.TUPLE([Types.INT(), Types.INT()])
).key_by(lambda x: x[0])

# Reduce: combine elements with same key
reduced = keyed.reduce(lambda a, b: (a[0], a[1] + b[1]))
```

### Windows

Windows divide streams into finite chunks for processing.

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.window import TumblingEventTimeWindows, SlidingEventTimeWindows
from pyflink.common.time import Time
from pyflink.common.typeinfo import Types

env = StreamExecutionEnvironment.get_execution_environment()

# Tumbling window (non-overlapping)
data_stream.key_by(lambda x: x[0]) \
    .window(TumblingEventTimeWindows.of(Time.seconds(10))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))

# Sliding window (overlapping)
data_stream.key_by(lambda x: x[0]) \
    .window(SlidingEventTimeWindows.of(Time.seconds(10), Time.seconds(5))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))

# Session window
from pyflink.datastream.window import EventTimeSessionWindows

data_stream.key_by(lambda x: x[0]) \
    .window(EventTimeSessionWindows.with_gap(Time.seconds(30))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
```

### State

State allows Flink to remember information across events.

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.functions import RuntimeContext, MapFunction
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.common.typeinfo import Types

class StatefulMapper(MapFunction):
    def open(self, runtime_context: RuntimeContext):
        descriptor = ValueStateDescriptor(
            "counter",
            Types.INT()
        )
        self.counter_state = runtime_context.get_state(descriptor)
    
    def map(self, value):
        current = self.counter_state.value()
        if current is None:
            current = 0
        current += 1
        self.counter_state.update(current)
        return (value, current)

env = StreamExecutionEnvironment.get_execution_environment()
data_stream = env.from_collection([1, 2, 3, 4, 5])
result = data_stream.key_by(lambda x: x % 2).map(StatefulMapper())
```



## Time and Watermarks

### Event Time

Event time is the time when an event actually occurred.

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.common.watermark_strategy import WatermarkStrategy
from pyflink.common.time import Duration
from pyflink.common.typeinfo import Types

env = StreamExecutionEnvironment.get_execution_environment()

# Set time characteristic
env.set_stream_time_characteristic(TimeCharacteristic.EventTime)

# Define watermark strategy
watermark_strategy = WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(5)) \
    .with_timestamp_assigner(lambda event, timestamp: event[1])

data_stream = env.from_collection(
    [(1, 1000), (2, 2000), (3, 1500)],
    type_info=Types.TUPLE([Types.INT(), Types.LONG()])
).assign_timestamps_and_watermarks(watermark_strategy)
```

### Processing Time

Processing time is the time when the operator processes the event.

```python
from pyflink.datastream import StreamExecutionEnvironment, TimeCharacteristic

env = StreamExecutionEnvironment.get_execution_environment()
env.set_stream_time_characteristic(TimeCharacteristic.ProcessingTime)

# Windows based on processing time
from pyflink.datastream.window import TumblingProcessingTimeWindows
from pyflink.common.time import Time

data_stream.key_by(lambda x: x[0]) \
    .window(TumblingProcessingTimeWindows.of(Time.seconds(10))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
```

### Watermarks

Watermarks track progress in event time and handle late events.

```python
from pyflink.common.watermark_strategy import WatermarkStrategy, TimestampAssigner
from pyflink.common.time import Duration

class CustomTimestampAssigner(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp):
        # Extract timestamp from value
        return value['timestamp']

watermark_strategy = WatermarkStrategy \
    .for_bounded_out_of_orderness(Duration.of_seconds(10)) \
    .with_timestamp_assigner(CustomTimestampAssigner())

data_stream = source_stream.assign_timestamps_and_watermarks(watermark_strategy)
```



## Sources and Sinks

### Built-in Sources

```python
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

# From collection
stream1 = env.from_collection([1, 2, 3, 4, 5])

# From elements
stream2 = env.from_elements(1, 2, 3, 4, 5)

# Socket source
stream3 = env.socket_text_stream("localhost", 9999)

# File source
stream4 = env.read_text_file("input.txt")
```

### Kafka Source

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors import FlinkKafkaConsumer
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
import json

env = StreamExecutionEnvironment.get_execution_environment()

properties = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'flink-consumer-group'
}

kafka_consumer = FlinkKafkaConsumer(
    topics='input-topic',
    deserialization_schema=SimpleStringSchema(),
    properties=properties
)

# Set starting position
kafka_consumer.set_start_from_earliest()
# kafka_consumer.set_start_from_latest()
# kafka_consumer.set_start_from_timestamp(1609459200000)

stream = env.add_source(kafka_consumer)
```

### Kafka Sink

```python
from pyflink.datastream.connectors import FlinkKafkaProducer
from pyflink.common.serialization import SimpleStringSchema

properties = {
    'bootstrap.servers': 'localhost:9092'
}

kafka_producer = FlinkKafkaProducer(
    topic='output-topic',
    serialization_schema=SimpleStringSchema(),
    producer_config=properties
)

# Write to Kafka
data_stream.map(lambda x: json.dumps(x)).add_sink(kafka_producer)
```

### File Sinks

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors import StreamingFileSink
from pyflink.common.serialization import Encoder

env = StreamExecutionEnvironment.get_execution_environment()

# Simple file sink
data_stream.write_as_text("output.txt")

# Bucketing file sink
sink = StreamingFileSink \
    .for_row_format("output_path", Encoder.simple_string_encoder()) \
    .build()

data_stream.add_sink(sink)
```



## Stateful Stream Processing

### ValueState

```python
from pyflink.datastream.functions import KeyedProcessFunction
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.common.typeinfo import Types

class CounterFunction(KeyedProcessFunction):
    def open(self, runtime_context):
        self.counter_state = runtime_context.get_state(
            ValueStateDescriptor("counter", Types.INT())
        )
    
    def process_element(self, value, ctx):
        current = self.counter_state.value()
        if current is None:
            current = 0
        current += 1
        self.counter_state.update(current)
        yield (value, current)

data_stream.key_by(lambda x: x[0]).process(CounterFunction())
```

### ListState

```python
from pyflink.datastream.functions import KeyedProcessFunction
from pyflink.datastream.state import ListStateDescriptor
from pyflink.common.typeinfo import Types

class BufferFunction(KeyedProcessFunction):
    def open(self, runtime_context):
        self.buffer_state = runtime_context.get_list_state(
            ListStateDescriptor("buffer", Types.INT())
        )
    
    def process_element(self, value, ctx):
        # Add to buffer
        self.buffer_state.add(value[1])
        
        # Get all buffered values
        buffered = list(self.buffer_state.get())
        
        # Output if buffer size reaches threshold
        if len(buffered) >= 10:
            yield (value[0], sum(buffered))
            self.buffer_state.clear()

data_stream.key_by(lambda x: x[0]).process(BufferFunction())
```

### MapState

```python
from pyflink.datastream.functions import KeyedProcessFunction
from pyflink.datastream.state import MapStateDescriptor
from pyflink.common.typeinfo import Types

class AggregatorFunction(KeyedProcessFunction):
    def open(self, runtime_context):
        self.map_state = runtime_context.get_map_state(
            MapStateDescriptor("aggregates", Types.STRING(), Types.INT())
        )
    
    def process_element(self, value, ctx):
        category = value[1]
        amount = value[2]
        
        # Get current value
        current = self.map_state.get(category)
        if current is None:
            current = 0
        
        # Update
        self.map_state.put(category, current + amount)
        
        # Output all categories
        result = {k: v for k, v in self.map_state.items()}
        yield (value[0], result)
```



## Checkpointing and Fault Tolerance

### Checkpoint Configuration

```python
from pyflink.datastream import StreamExecutionEnvironment, CheckpointingMode
from pyflink.common.time import Time

env = StreamExecutionEnvironment.get_execution_environment()

# Enable checkpointing
env.enable_checkpointing(60000)  # Every 60 seconds

# Set checkpoint mode
env.get_checkpoint_config().set_checkpointing_mode(CheckpointingMode.EXACTLY_ONCE)

# Set minimum pause between checkpoints
env.get_checkpoint_config().set_min_pause_between_checkpoints(30000)

# Set checkpoint timeout
env.get_checkpoint_config().set_checkpoint_timeout(600000)

# Set max concurrent checkpoints
env.get_checkpoint_config().set_max_concurrent_checkpoints(1)

# Enable externalized checkpoints
from pyflink.datastream import ExternalizedCheckpointCleanup

env.get_checkpoint_config().enable_externalized_checkpoints(
    ExternalizedCheckpointCleanup.RETAIN_ON_CANCELLATION
)

# Set checkpoint storage
env.get_checkpoint_config().set_checkpoint_storage("file:///checkpoint-dir")
```

### Savepoints

```bash
# Trigger savepoint
flink savepoint <jobId> /path/to/savepoint

# Cancel job with savepoint
flink cancel -s /path/to/savepoint <jobId>

# Resume from savepoint
flink run -s /path/to/savepoint job.jar
```



## Window Operations

### Tumbling Windows

```python
from pyflink.datastream.window import TumblingEventTimeWindows
from pyflink.common.time import Time

# 10-second tumbling window
windowed = data_stream \
    .key_by(lambda x: x[0]) \
    .window(TumblingEventTimeWindows.of(Time.seconds(10))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
```

### Sliding Windows

```python
from pyflink.datastream.window import SlidingEventTimeWindows
from pyflink.common.time import Time

# 10-second window, sliding every 5 seconds
windowed = data_stream \
    .key_by(lambda x: x[0]) \
    .window(SlidingEventTimeWindows.of(Time.seconds(10), Time.seconds(5))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
```

### Session Windows

```python
from pyflink.datastream.window import EventTimeSessionWindows
from pyflink.common.time import Time

# Session window with 30-second gap
windowed = data_stream \
    .key_by(lambda x: x[0]) \
    .window(EventTimeSessionWindows.with_gap(Time.seconds(30))) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
```

### Custom Windows

```python
from pyflink.datastream.functions import ProcessWindowFunction
from pyflink.datastream.window import TimeWindow

class CustomWindowFunction(ProcessWindowFunction):
    def process(self, key, context, elements):
        count = 0
        sum_val = 0
        for element in elements:
            count += 1
            sum_val += element[1]
        
        window = context.window()
        yield (
            key,
            sum_val,
            count,
            window.start,
            window.end
        )

windowed = data_stream \
    .key_by(lambda x: x[0]) \
    .window(TumblingEventTimeWindows.of(Time.seconds(10))) \
    .process(CustomWindowFunction())
```



## Advanced Features

### Side Outputs

```python
from pyflink.datastream import OutputTag
from pyflink.datastream.functions import ProcessFunction

# Define output tags
high_output = OutputTag("high", Types.TUPLE([Types.INT(), Types.INT()]))
low_output = OutputTag("low", Types.TUPLE([Types.INT(), Types.INT()]))

class SplitFunction(ProcessFunction):
    def process_element(self, value, ctx):
        if value[1] > 50:
            yield high_output, value
        else:
            yield low_output, value

main_stream = data_stream.process(SplitFunction())
high_stream = main_stream.get_side_output(high_output)
low_stream = main_stream.get_side_output(low_output)
```

### Async I/O

```python
from pyflink.datastream.functions import AsyncFunction
import asyncio

class AsyncDatabaseLookup(AsyncFunction):
    async def async_invoke(self, input_value, result_future):
        try:
            # Async database query
            result = await self.query_database(input_value)
            result_future.complete([result])
        except Exception as e:
            result_future.complete_exceptionally(e)
    
    async def query_database(self, value):
        # Simulated async database call
        await asyncio.sleep(0.1)
        return value * 2

# Apply async function
async_stream = AsyncDataStream.unordered_wait(
    data_stream,
    AsyncDatabaseLookup(),
    timeout=10000,
    capacity=100
)
```

### Broadcast State

```python
from pyflink.datastream.functions import BroadcastProcessFunction
from pyflink.datastream.state import MapStateDescriptor
from pyflink.common.typeinfo import Types

# Define broadcast state descriptor
config_descriptor = MapStateDescriptor(
    "config",
    Types.STRING(),
    Types.STRING()
)

# Broadcast stream
config_stream = env.from_collection([("threshold", "100")])
broadcast_stream = config_stream.broadcast(config_descriptor)

class ConfigurableProcessor(BroadcastProcessFunction):
    def process_element(self, value, ctx):
        config = ctx.get_broadcast_state(config_descriptor)
        threshold = int(config.get("threshold") or 0)
        
        if value[1] > threshold:
            yield value
    
    def process_broadcast_element(self, value, ctx):
        config = ctx.get_broadcast_state(config_descriptor)
        config.put(value[0], value[1])

result = data_stream.connect(broadcast_stream).process(ConfigurableProcessor())
```



## Table API and SQL

### Table API

```python
from pyflink.table import EnvironmentSettings, TableEnvironment

# Create table environment
env_settings = EnvironmentSettings.in_streaming_mode()
table_env = TableEnvironment.create(env_settings)

# Create table from DataStream
table = table_env.from_data_stream(
    data_stream,
    ['user_id', 'amount', 'timestamp']
)

# Apply transformations
result_table = table.select(table.user_id, table.amount) \
    .where(table.amount > 100) \
    .group_by(table.user_id) \
    .select(table.user_id, table.amount.sum.alias('total'))

# Convert back to DataStream
result_stream = table_env.to_data_stream(result_table)
```

### SQL Queries

```python
from pyflink.table import EnvironmentSettings, TableEnvironment

table_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

# Register table
table_env.execute_sql("""
    CREATE TABLE source_table (
        user_id INT,
        amount DOUBLE,
        ts TIMESTAMP(3),
        WATERMARK FOR ts AS ts - INTERVAL '5' SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'input-topic',
        'properties.bootstrap.servers' = 'localhost:9092',
        'format' = 'json'
    )
""")

# Execute SQL query
result = table_env.sql_query("""
    SELECT 
        user_id,
        SUM(amount) as total_amount,
        COUNT(*) as transaction_count,
        TUMBLE_START(ts, INTERVAL '1' MINUTE) as window_start
    FROM source_table
    GROUP BY user_id, TUMBLE(ts, INTERVAL '1' MINUTE)
    HAVING SUM(amount) > 1000
""")

# Create sink table
table_env.execute_sql("""
    CREATE TABLE sink_table (
        user_id INT,
        total_amount DOUBLE,
        transaction_count BIGINT,
        window_start TIMESTAMP(3)
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'output-topic',
        'properties.bootstrap.servers' = 'localhost:9092',
        'format' = 'json'
    )
""")

# Insert into sink
result.execute_insert('sink_table')
```



## Performance Optimization

### Parallelism

```python
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

# Set global parallelism
env.set_parallelism(4)

# Set operator-specific parallelism
data_stream.map(lambda x: x * 2).set_parallelism(8)

# Disable parallelism for specific operator
data_stream.map(lambda x: x * 2).set_parallelism(1)
```

### Resource Management

```python
# Configure task slots
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

# Set buffer timeout
env.set_buffer_timeout(100)

# Configure network buffers
config = env.get_config()
config.set_task_cancellation_timeout(30000)
```

### State Backend Configuration

```python
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

# HashMapStateBackend (formerly MemoryStateBackend)
env.set_state_backend("hashmap")

# RocksDB state backend for large state
env.set_state_backend("rocksdb")

# Configure RocksDB
from pyflink.datastream.state_backend import RocksDBStateBackend

state_backend = RocksDBStateBackend("file:///checkpoint-dir", True)
env.set_state_backend(state_backend)
```



## Best Practices

### Exactly-Once Processing

```python
from pyflink.datastream import StreamExecutionEnvironment, CheckpointingMode

env = StreamExecutionEnvironment.get_execution_environment()

# Configure for exactly-once
env.enable_checkpointing(60000)
env.get_checkpoint_config().set_checkpointing_mode(CheckpointingMode.EXACTLY_ONCE)
env.get_checkpoint_config().set_min_pause_between_checkpoints(30000)

# Use idempotent sinks
# Ensure Kafka producer has transactions enabled
```

### Handling Late Data

```python
from pyflink.datastream.window import TumblingEventTimeWindows
from pyflink.common.time import Time

# Allow late data up to 1 minute
windowed = data_stream \
    .key_by(lambda x: x[0]) \
    .window(TumblingEventTimeWindows.of(Time.minutes(1))) \
    .allowed_lateness(Time.minutes(1)) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))

# Handle late data with side output
late_output_tag = OutputTag("late-data", Types.TUPLE([Types.INT(), Types.INT()]))

windowed_with_late = data_stream \
    .key_by(lambda x: x[0]) \
    .window(TumblingEventTimeWindows.of(Time.minutes(1))) \
    .allowed_lateness(Time.minutes(1)) \
    .side_output_late_data(late_output_tag) \
    .reduce(lambda a, b: (a[0], a[1] + b[1]))

late_data_stream = windowed_with_late.get_side_output(late_output_tag)
```

### Memory Management

```python
# Configure memory for task managers
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

# Set managed memory for state backend
config = env.get_config()

# For RocksDB state backend
env.set_state_backend("rocksdb")

# Configure incremental checkpoints for RocksDB
checkpoint_config = env.get_checkpoint_config()
checkpoint_config.enable_unaligned_checkpoints()
```

### Monitoring and Debugging

```python
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

# Enable metrics
env.get_config().set_latency_tracking_interval(1000)

# Add custom metrics
from pyflink.datastream.functions import MapFunction

class MetricMapper(MapFunction):
    def open(self, runtime_context):
        self.counter = runtime_context.get_metrics_group() \
            .counter("processed_records")
    
    def map(self, value):
        self.counter.inc()
        return value

data_stream.map(MetricMapper())
```